# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/real-huzaifa/flyrank-internship-ml-track/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2: Refresh / Content Opportunity Scoring, Task type: scoring / ranking**

The output is an ordered queue an editor works down. Underneath it is a binary classifier
estimating P(page is declining); I rank by that probability. So: a ranking task served by a
classifier, not a learning-to-rank model, and not a classifier whose yes/no anyone sees.

**Why not classification as the deliverable.** Simple triggers already fire on 13,191 of 30,000
eligible pages — 44% of the corpus, about five years of work at 50 reviews a week. A yes/no that
says "yes" to four pages in ten hands the editor a pile, not a decision. Detection is cheap;
editor-hours are scarce.

**Why not clustering.** It would tell me what kinds of pages exist. That informs a content audit
but produces no order, and order is the deliverable.

**Why not signal analysis.** It answers which signals travel together. Useful input, but nothing
an editor can act on.

The decision is *which page do I open first*. That is an ordering question, so the output is an
order.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

REPO = "flyrank-internship-ml-track"

# Colab loads only the .ipynb from GitHub, not the repo — clone if the data is missing.
if not Path("data/raw").exists():
    if not Path(REPO).exists():
        os.system(f"git clone https://github.com/real-huzaifa/{REPO}.git")
    os.chdir(REPO)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The starter pipeline's own eligibility filter (scripts/01_prepare_features.py)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

WEEKLY_CAPACITY = 50

stale     = (eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)
declining = (eligible["trend_direction"] == "down")     & (eligible["impressions_90d"] >= 100)
thin      = (eligible["word_count"] > 0) & (eligible["word_count"] < 1200) & (eligible["impressions_90d"] >= 250)
flagged   = (stale | declining | thin).sum()

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns | {df['client_id'].nunique()} clients")
print(f"Eligible for review: {len(eligible):,}")
print()
print("WHY THIS IS A RANKING TASK, NOT A DETECTION TASK")
print(f"  Pages flagged by simple triggers  : {flagged:,}")
print(f"  Review capacity                   : {WEEKLY_CAPACITY}/week")
print(f"  Time to clear the flagged pool    : {flagged/WEEKLY_CAPACITY:.0f} weeks (~{flagged/WEEKLY_CAPACITY/52:.1f} years)")
print(f"  Fraction of eligible pages flagged: {flagged/len(eligible)*100:.1f}%")
print()
print("  A binary flag that fires on 4 pages in 10 gives the editor no way to choose.")
print("  The scarce resource is editor-hours, so the output must be an ORDER, not a yes/no.")

Loaded: 30,000 rows x 44 columns | 32 clients
Eligible for review: 30,000

WHY THIS IS A RANKING TASK, NOT A DETECTION TASK
  Pages flagged by simple triggers  : 13,191
  Review capacity                   : 50/week
  Time to clear the flagged pool    : 264 weeks (~5.1 years)
  Fraction of eligible pages flagged: 44.0%

  A binary flag that fires on 4 pages in 10 gives the editor no way to choose.
  The scarce resource is editor-hours, so the output must be an ORDER, not a yes/no.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I would predict:** whether a page's organic search traffic is about to decline, expressed
as a probability so the queue can be ranked by it.

**Where the label comes from: a defined rule, not an observed outcome.** The only label here is
`is_declining_label` (`scripts/01_prepare_features.py:110`), which is `trend_direction == "down"`.
Its chain:

`impressions_last_30d` + `impressions_prev_30d` → `trend_pct` → `trend_direction` → `is_declining_label`

The cell below reconstructs `trend_pct` exactly from those two raw columns in 100% of rows
(correlation 1.0000). A model trained on this learns the rule, not the world. It is also a
current-window bucket: it describes what already happened and says nothing about the future.

A second defect: 3,388 pages (11.3%) are marked `new` or `flat`, meaning the trend could not be
computed. The label files all of them as negatives.

**The target I want** is an observed outcome in a later window:

> `y = 1` if a page's impressions over days `[t+1, t+30]` fall more than X% below its own
> `[t-30, t]` baseline, with features drawn only from `[t-90, t]`.

**Why not here.** The cell below scans all 44 columns: no datetime column, and no text column
parses as a date. This file is one snapshot with pre-baked windows, so there is no time axis and
no future window to measure. The real target needs `fact_content_daily_performance`
(78.8M rows at `report_date × client × content`) — ML-04 work.

**Position for now:** the proxy is a scaffold. I model against it to find out whether any
learnable signal exists and to set an honest floor, not because it is the target.

In [2]:
import warnings

print("[A] DOES THIS FILE CONTAIN ANY DATE / TIME COLUMN?")
dt_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
parseable = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for c in text_cols:
        try:
            pd.to_datetime(df[c].dropna().head(200), errors="raise")
            parseable.append(c)
        except Exception:
            pass
print(f"    datetime-typed columns          : {dt_cols}")
print(f"    text columns parseable as dates : {parseable}")
print("    => No timestamp anywhere. This file is ONE SNAPSHOT with pre-computed windows.")
print("       A future-window label cannot be built here. Not a preference — a hard limit.")
print()

print("[B] THE ONLY LABEL AVAILABLE: is_declining_label")
print("    scripts/01_prepare_features.py:110")
print('      is_declining_label = (trend_direction.lower() == "down")')
y_proxy = eligible["trend_direction"].str.lower().eq("down").astype(int)
print(f"    base rate: {y_proxy.mean()*100:.1f}%  ({y_proxy.sum():,} positives / {(1-y_proxy).sum():,} negatives)")
print()
print("    trend_direction distribution:")
for k, v in eligible["trend_direction"].value_counts().items():
    print(f"      {k:8s} {v:6,}  ({v/len(eligible)*100:4.1f}%)")
print('    => "new" and "flat" mean trend UNKNOWN, but the label files them as negatives.')
print()

print("[C] IS THAT LABEL OBSERVED, OR DEFINED BY A RULE?")
aud = df[(df["impressions_prev_30d"] > 0) & df["trend_pct"].notna()].copy()
aud["recon"] = (aud["impressions_last_30d"] - aud["impressions_prev_30d"]) / aud["impressions_prev_30d"] * 100
exact = (aud["trend_pct"] - aud["recon"]).abs() < 0.5
print(f"    trend_pct reconstructed exactly from the two 30d windows: {exact.mean()*100:.1f}% of {len(aud):,} rows")
print(f"    correlation: {aud['trend_pct'].corr(aud['recon']):.4f}")
print("    => DEFINED. The chain is:")
print("       impressions_last_30d + impressions_prev_30d -> trend_pct -> trend_direction -> label")
print()

print("[D] THE 90-DAY AGGREGATES CONTAIN BOTH LABEL WINDOWS")
for base, last, prev in [("impressions_90d", "impressions_last_30d", "impressions_prev_30d"),
                         ("clicks_90d",      "clicks_last_30d",      "clicks_prev_30d"),
                         ("sessions_90d",    "sessions_last_30d",    "sessions_prev_30d")]:
    s = df[last] + df[prev]
    share = (s / df[base].replace(0, np.nan)).median()
    print(f"    {base:16s} >= last30+prev30 in {(df[base] >= s).mean()*100:5.1f}% of rows"
          f" | median share of the 90d total: {share*100:.1f}%")
print("    => These are NOT clean features for this label. They are PARTIAL leakage.")
print("       Any score built on them is scored against a target it partly contains.")

[A] DOES THIS FILE CONTAIN ANY DATE / TIME COLUMN?
    datetime-typed columns          : []
    text columns parseable as dates : []
    => No timestamp anywhere. This file is ONE SNAPSHOT with pre-computed windows.
       A future-window label cannot be built here. Not a preference — a hard limit.

[B] THE ONLY LABEL AVAILABLE: is_declining_label
    scripts/01_prepare_features.py:110
      is_declining_label = (trend_direction.lower() == "down")
    base rate: 54.2%  (16,262 positives / 13,738 negatives)

    trend_direction distribution:
      down     16,262  (54.2%)
      stable    5,962  (19.9%)
      up        4,388  (14.6%)
      new       2,236  ( 7.5%)
      flat      1,152  ( 3.8%)
    => "new" and "flat" mean trend UNKNOWN, but the label files them as negatives.

[C] IS THAT LABEL OBSERVED, OR DEFINED BY A RULE?
    trend_pct reconstructed exactly from the two 30d windows: 100.0% of 26,612 rows
    correlation: 1.0000
    => DEFINED. The chain is:
       impressions_last_30

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50**, reported alongside average precision and a by-hand read of the top 20.

K = 50 is the team's weekly review capacity. Precision@50 answers the only question the editor
experiences: of the 50 pages I open this week, how many deserved my time. Recall over 30,000
pages is irrelevant to someone who never sees page 51.

Not accuracy: the base rate is 54.2%, so predicting "declining" for everything scores 54% and
helps nobody.

**What "good" means, fixed before any modelling:**

| level | P@50 |
|---|---|
| floor — must clear | > 0.542 (beats random) |
| useful | ≥ 0.70 |
| strong | ≥ 0.80 |

The floor comes from measurement, not intuition. The cell below shows random ordering gets
P@50 = 0.542, and the starter pipeline's `baseline_refresh_score` gets **0.340** — below the 5th
percentile of 500 random draws. The shipped baseline ranks worse than shuffling the pages.

The mechanism is in the formula (`scripts/02_baseline_score.py:62-76`): 40% of the weight is
visibility, 30% is staleness. It ranks big old pages, and big old pages are not the declining
ones (section 5 shows `content_age_days` correlates negatively with the label).

Note the split verdict: average precision is 0.570 against a 0.542 base rate, so across the whole
ranking the rule is marginally better than chance. It fails specifically at the top — the only
part anyone uses. That is why precision@K and not a global metric.

These thresholds are set against the proxy label. When the real target arrives the base rate
changes and every number here has to be re-derived; what carries over is the method.

In [3]:
from sklearn.metrics import average_precision_score

work = eligible.copy()
work["y"] = work["trend_direction"].str.lower().eq("down").astype(int)


def pct_rank(s):
    return s.rank(pct=True).fillna(0.0)


def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0.0


# Reproduce the starter baseline exactly (scripts/02_baseline_score.py:62-76)
work["visibility_score"] = pct_rank(np.log1p(work["impressions_90d"]))
work["freshness_risk_score"] = pct_rank(work["days_since_last_update"])
work["position_opportunity_score"] = (
    (1 - minmax(work["avg_position"].clip(lower=1, upper=50)))
    * work["visibility_score"]
    * (work["avg_position"] > 0).astype(int)
)
work["depth_gap_score"] = (1 - pct_rank(work["word_count"])) * work["visibility_score"]
work["baseline_refresh_score"] = (
    0.40 * work["visibility_score"]
    + 0.30 * work["freshness_risk_score"]
    + 0.25 * work["position_opportunity_score"]
    + 0.05 * work["depth_gap_score"]
).clip(0, 1)

base_rate = work["y"].mean()

# Random-ordering reference: 500 draws of 50 pages
rng = np.random.default_rng(42)
rand_p50 = np.array([work["y"].to_numpy()[rng.choice(len(work), 50, replace=False)].mean()
                     for _ in range(500)])

print("METRIC: precision@50 — the share of the top 50 ranked pages that are true positives.")
print("Chosen because the editor only ever sees the top of the queue.\n")
print(f"{'ranker':32s} {'P@50':>8s} {'P@20':>8s} {'P@100':>8s}")
print("-" * 60)
ranked = work.sort_values("baseline_refresh_score", ascending=False)
print(f"{'starter baseline_refresh_score':32s} "
      + " ".join(f"{ranked.head(K)['y'].mean():8.3f}" for K in (50, 20, 100)))
print(f"{'random ordering (mean of 500)':32s} {rand_p50.mean():8.3f} {'—':>8s} {'—':>8s}")
print(f"{'base rate (predict-all-positive)':32s} {base_rate:8.3f}")
print("-" * 60)
print(f"random P@50 5th–95th percentile: {np.percentile(rand_p50,5):.3f} – {np.percentile(rand_p50,95):.3f}")
print()

starter_p50 = work.nlargest(50, "baseline_refresh_score")["y"].mean()
print("READ THIS HONESTLY:")
print(f"  The existing hand-written score gets P@50 = {starter_p50:.3f}.")
print(f"  Random ordering gets {rand_p50.mean():.3f}.")
print(f"  The rule beats only {(rand_p50 < starter_p50).mean()*100:.1f}% of random draws — it is WORSE than chance.")
print()
print("TARGETS SET BEFORE ANY MODELLING:")
print(f"  floor (must clear)  : P@50 > {base_rate:.3f}   [beat random]")
print(f"  useful              : P@50 >= 0.70")
print(f"  strong              : P@50 >= 0.80")
print("  Reported alongside average precision and a by-hand read of the top 20.")
print(f"\n  average precision of the starter baseline: "
      f"{average_precision_score(work['y'], work['baseline_refresh_score']):.3f} (base rate {base_rate:.3f})")

METRIC: precision@50 — the share of the top 50 ranked pages that are true positives.
Chosen because the editor only ever sees the top of the queue.

ranker                               P@50     P@20    P@100
------------------------------------------------------------
starter baseline_refresh_score      0.340    0.350    0.370
random ordering (mean of 500)       0.542        —        —
base rate (predict-all-positive)    0.542
------------------------------------------------------------
random P@50 5th–95th percentile: 0.420 – 0.660

READ THIS HONESTLY:
  The existing hand-written score gets P@50 = 0.340.
  Random ordering gets 0.542.
  The rule beats only 0.0% of random draws — it is WORSE than chance.

TARGETS SET BEFORE ANY MODELLING:
  floor (must clear)  : P@50 > 0.542   [beat random]
  useful              : P@50 >= 0.70
  strong              : P@50 >= 0.80
  Reported alongside average precision and a by-hand read of the top 20.

  average precision of the starter baseline: 0.570

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page, belonging to one client, observed at one moment in time.**

The key is `(client_id, content_id)`. The grain probe below confirms it: 30,000 rows, 30,000
distinct keys, zero duplicates. Both IDs are pseudonyms — grouping, joining and splitting only,
never features.

Each row carries search-demand context (`search_volume`, `competition`, `cpc`), content
properties (`content_type`, `word_count`, `content_age_days`, `days_since_last_update`), and
90-day performance (`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `engagement_rate`,
`scroll_rate`).

The model emits one priority score per row, which becomes one rank in the queue. Unit of
prediction = unit of action = unit of analysis.

**What the grain forces on me.** Pages are unevenly spread across clients: 3 for the smallest,
7,008 for the largest, which alone is 23.4% of the corpus. A random row-level split would put
pages from the same client on both sides and let the model memorise client quirks. Validation
must be `GroupKFold(groups=client_id)`.

**What the grain is missing.** The unit is right; the time axis is absent.
`fact_content_daily_performance` has grain `report_date × client × content`, which supports
features from `[t-90, t]` and a label from `[t+1, t+30]`. This file collapses that to a snapshot.
The unit I need in ML-04 is `(client_id, content_id, as_of_date)` — same page-level unit, plus
the dimension that makes a real target possible.

In [4]:
print("[A] GRAIN PROBE — is (client_id, content_id) really unique?")
dupes = df.groupby(["client_id", "content_id"]).size()
print(f"    rows                       : {len(df):,}")
print(f"    distinct (client, content) : {len(dupes):,}")
print(f"    keys appearing >1 time     : {(dupes > 1).sum()}")
print(f"    distinct content_id alone  : {df['content_id'].nunique():,}")
print("    => grain confirmed: ONE ROW = ONE CONTENT PAGE, at one moment in time.")
print()

print("[B] WHAT ONE ROW ACTUALLY LOOKS LIKE (the unit of analysis)")
unit_cols = ["content_id", "client_id", "content_type",
             "impressions_90d", "clicks_90d", "ctr", "avg_position",
             "content_age_days", "days_since_last_update", "word_count"]
unit = eligible[unit_cols].head(5)
try:
    display(unit)
except NameError:
    print(unit.to_string(index=False))
print()

print("[C] THE SCORING FRAME — what the model consumes and emits")
scoring_frame = work[["content_id", "client_id"]
                     + [c for c in unit_cols if c not in ("content_id", "client_id")]].copy()
scoring_frame["baseline_refresh_score"] = work["baseline_refresh_score"]
scoring_frame["y_proxy_declining"] = work["y"]
print(f"    shape         : {scoring_frame.shape[0]:,} rows x {scoring_frame.shape[1]} columns")
print(f"    key           : (client_id, content_id)")
print(f"    clients       : {scoring_frame['client_id'].nunique()}")
print(f"    output per row: one priority score -> one rank in the review queue")
print()
print(scoring_frame.head(3).to_string(index=False))
print()

print("[D] PAGES PER CLIENT — why splits must be grouped by client")
per_client = eligible.groupby("client_id").size().sort_values(ascending=False)
print(f"    min / median / max pages per client: {per_client.min():,} / {int(per_client.median()):,} / {per_client.max():,}")
print(f"    top client holds {per_client.iloc[0]/len(eligible)*100:.1f}% of all rows")
print("    => a random row split leaks client-specific patterns across train/test.")
print("       Validation must use GroupKFold(groups=client_id).")
print()

print("[E] THE GRAIN I ACTUALLY NEED (warehouse, ML-04+)")
print("    fact_content_daily_performance: report_date x client x content, 78,835,655 rows")
print("    That grain supports: features from days [t-90, t], label from days [t+1, t+30].")
print("    THIS file collapses that to a single snapshot, so the real target is not")
print("    constructible here. The unit of analysis is right; the time axis is missing.")

[A] GRAIN PROBE — is (client_id, content_id) really unique?
    rows                       : 30,000
    distinct (client, content) : 30,000
    keys appearing >1 time     : 0
    distinct content_id alone  : 30,000
    => grain confirmed: ONE ROW = ONE CONTENT PAGE, at one moment in time.

[B] WHAT ONE ROW ACTUALLY LOOKS LIKE (the unit of analysis)


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20,3221.0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,3515.0
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14,2803.0



[C] THE SCORING FRAME — what the model consumes and emits
    shape         : 30,000 rows x 12 columns
    key           : (client_id, content_id)
    clients       : 32
    output per row: one priority score -> one rank in the review queue

          content_id         client_id    content_type  impressions_90d  clicks_90d  ctr  avg_position  content_age_days  days_since_last_update  word_count  baseline_refresh_score  y_proxy_declining
content_304f48230142 client_f369cb89fc keyword article             3803          29 0.76          10.6               187                      20      3221.0                0.568858                  1
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7 0.05          20.3               445                      25      2481.0                0.737959                  1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11 0.09          36.5               141                      20      3515.0 

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two things have to hold: the pattern must be real, and it must be too messy to hand-write. I
tested both.

**Too messy to hand-write.** The strongest single correlation with the label across all 34
non-leaky features is `content_age_days` at ρ = −0.158. No feature exceeds |ρ| = 0.20; only five
clear 0.10. There is no variable strong enough to hang a threshold on.

The direction matters too. `content_age_days` correlates *negatively* — older pages are somewhat
less likely to be labelled declining — yet the starter rule gives staleness 30% weight and pushes
stale pages up. That sign error explains most of its 0.340 P@50. Weights guessed by hand got the
direction backwards on the second-heaviest term; weights learned from data would not.

**The pattern is real.** Under `GroupKFold(n_splits=5, groups=client_id)`, gradient boosting on
the same non-leaky features reaches **P@50 = 0.824** and ROC-AUC 0.679, against 0.340 for the rule
and 0.542 for random. It clears my pre-set "strong" threshold on clients it never saw.

**What limits that number.** Three of the features — `impressions_90d`, `clicks_90d`,
`sessions_90d` — partly contain the windows that define the label: each is ≥ the sum of its two
30-day components in 100% of rows, with those components making up 57–77% of the total. So the
result shows that a learned combination of weak signals orders this queue far better than a
hand-written rule. It does not show that anything predicts future decline. ROC-AUC 0.679 is an
upper bound on a contaminated task, and I expect a lower number on a proper `[t+1, t+30]` label.

Fold variance is wide: P@50 ranges 0.680–0.940, because GroupKFold gives the largest client its
own fold. The mean is directional, not precise.

**Claim:** on this snapshot, against a proxy label, learned weights beat hand-set weights by a
wide margin, and the reason is visible — the signal is spread thin across many features and at
least one hand-set weight points the wrong way. Whether ML beats a rule on the real forecasting
task is open, and belongs to ML-04 through ML-08.

In [5]:
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

LEAKY = ["trend_direction", "trend_pct",
         "impressions_last_30d", "impressions_prev_30d",
         "clicks_last_30d", "clicks_prev_30d",
         "sessions_last_30d", "sessions_prev_30d"]
IDS = ["content_id", "client_id"]
DERIVED = ["y", "visibility_score", "freshness_risk_score",
           "position_opportunity_score", "depth_gap_score", "baseline_refresh_score"]

features = [c for c in work.columns if c not in LEAKY + IDS + DERIVED]
num_feats = [c for c in features if pd.api.types.is_numeric_dtype(work[c])]
cat_feats = [c for c in features if c not in num_feats]

X = work[num_feats].copy()
for c in cat_feats:
    X[c] = pd.Categorical(work[c]).codes      # -1 encodes missing, kept as its own value
y = work["y"].to_numpy()
groups = work["client_id"].to_numpy()

print(f"Features used: {len(num_feats)} numeric + {len(cat_feats)} categorical = {X.shape[1]}")
print(f"Excluded as leaky: {LEAKY}")
print()

print("[A] IS ANY SINGLE SIGNAL STRONG ENOUGH TO WRITE A RULE FROM?")
rows = []
for c in num_feats:
    s = work[c]
    if s.nunique() < 2:
        continue
    mask = s.notna()
    rho, p = spearmanr(s[mask], work["y"][mask])
    rows.append((c, rho, p))
sig = (pd.DataFrame(rows, columns=["feature", "spearman_rho", "p_value"])
         .assign(abs_rho=lambda d: d["spearman_rho"].abs())
         .sort_values("abs_rho", ascending=False)
         .reset_index(drop=True))
print(sig.head(12).round(4).to_string(index=False))
print()
print(f"    strongest single correlation : |rho| = {sig['abs_rho'].max():.3f}  ({sig.iloc[0]['feature']})")
print(f"    features with |rho| > 0.20   : {(sig['abs_rho'] > 0.20).sum()}")
print(f"    features with |rho| > 0.10   : {(sig['abs_rho'] > 0.10).sum()}")
print("    => No dominant signal. There is no single threshold worth hard-coding.")
print("       Note the SIGN on content_age_days: older pages are LESS likely to be labelled")
print("       declining, yet the starter rule gives staleness 30% weight pushing them UP.")
print()

print("[B] DOES COMBINING THEM ACTUALLY BEAT THE RULE? (GroupKFold by client_id, 5 folds)")
aucs, aps, p50s = [], [], []
for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X, y, groups), start=1):
    model = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(X.iloc[tr], y[tr])
    proba = model.predict_proba(X.iloc[te])[:, 1]
    top50 = np.argsort(-proba)[:50]
    aucs.append(roc_auc_score(y[te], proba))
    aps.append(average_precision_score(y[te], proba))
    p50s.append(y[te][top50].mean())
    print(f"    fold {fold}: n_test={len(te):6,}  held-out clients={len(set(groups[te])):2d}  "
          f"ROC-AUC={aucs[-1]:.3f}  AP={aps[-1]:.3f}  P@50={p50s[-1]:.3f}")

print()
print(f"{'ranker':34s} {'P@50':>8s} {'ROC-AUC':>9s} {'AP':>8s}")
print("-" * 62)
print(f"{'model (grouped CV mean)':34s} {np.mean(p50s):8.3f} {np.mean(aucs):9.3f} {np.mean(aps):8.3f}")
print(f"{'starter rule baseline':34s} {starter_p50:8.3f} {'—':>9s} "
      f"{average_precision_score(work['y'], work['baseline_refresh_score']):8.3f}")
print(f"{'random ordering':34s} {rand_p50.mean():8.3f} {0.500:9.3f} {base_rate:8.3f}")
print("-" * 62)
print(f"model P@50 spread across folds: {min(p50s):.3f} – {max(p50s):.3f}")
print()

print("[C] WHAT THIS DOES AND DOES NOT SHOW")
print(f"    DOES : combining {X.shape[1]} weak signals orders the queue far better than the")
print(f"           hand-written rule ({np.mean(p50s):.3f} vs {starter_p50:.3f} P@50), and it holds up on")
print("           clients the model never saw. Complexity earns its keep here.")
print("    DOES NOT: predict future decline. The target is a CURRENT-window bucket, and")
print("           impressions_90d / clicks_90d / sessions_90d partly contain the very")
print(f"           windows that define it (section 2[D]). So ROC-AUC {np.mean(aucs):.3f} is an")
print("           UPPER bound on a contaminated task, not a forecast score.")
print("    NEXT : rebuild on the warehouse daily grain with a [t+1, t+30] label,")
print("           re-run the leakage audit, and re-earn every number above.")

Features used: 23 numeric + 11 categorical = 34
Excluded as leaky: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']

[A] IS ANY SINGLE SIGNAL STRONG ENOUGH TO WRITE A RULE FROM?
               feature  spearman_rho  p_value  abs_rho
      content_age_days       -0.1579      0.0   0.1579
        age_tier_order       -0.1570      0.0   0.1570
       impressions_90d        0.1458      0.0   0.1458
 days_with_impressions        0.1405      0.0   0.1405
         search_volume       -0.1091      0.0   0.1091
            word_count        0.0794      0.0   0.0794
            char_count        0.0684      0.0   0.0684
          avg_position        0.0509      0.0   0.0509
days_since_last_update        0.0495      0.0   0.0495
            clicks_90d        0.0452      0.0   0.0452
         pageviews_90d        0.0447      0.0   0.0447
           competition       -0.0374      0.0   0.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.